In [32]:
%pip install sympy pandas


Note: you may need to restart the kernel to use updated packages.


In [33]:
import sympy as sp
import json
import pandas as pd

# define symbolic variable
x = sp.symbols('x')


In [34]:
with open("quadratic_dataset.json", "r") as f:
    dataset = json.load(f)

print("Total entries:", len(dataset))


Total entries: 2100


In [35]:
dataset[0]

{'equation': '1x^2 + -6x + 0 = 0',
 'steps': ['1x^2 + -6x + 0 = 0',
  '(x - 6)(x - 0) = 0',
  'x - 6 = 0 OR x - 0 = 0',
  'x = 6 OR x = 0']}

In [36]:
import re

def normalize_expression(expr):
    # 1. Standardize powers
    expr = expr.replace("^", "**")
    
    # 2. Fix implied multiplication: (x-2)(x-3) -> (x-2)*(x-3)
    # We use r'\)\(' to match a literal ')' followed by a literal '('
    expr = re.sub(r'\)\(', r')*(', expr)
    
    # 3. Fix 5x -> 5*x
    expr = re.sub(r'(\d)([a-z])', r'\1*\2', expr)
    
    return expr

In [37]:
def sympy_digitize(expr):

    expr = normalize_expression(expr)

    try:
        return sp.sympify(expr)

    except:
        return None


In [38]:
def digitize_equation(equation):
    # Safely handle cases without an equals sign
    if "=" in equation:
        left = equation.split("=")[0]
    else:
        left = equation
        
    return sympy_digitize(left)

# Always test with a print to see what's happening under the hood
test_val = "(x-2)(x-3)"
print(f"Testing {test_val} -> {normalize_expression(test_val)}")

Testing (x-2)(x-3) -> (x-2)*(x-3)


In [39]:
def digitize_steps(steps):
    sympy_steps = []
    for step in steps:
        # 1. Split to get the left side of the equation
        left = step.split("=")[0]
        
        # 2. Clean the string to remove "OR" or "or"
        clean_step = re.sub(r'\s*or\s*', '', left, flags=re.IGNORECASE)
        
        # 3. Digitization
        sympy_expr = sympy_digitize(clean_step)
        sympy_steps.append(sympy_expr)
    return sympy_steps


In [40]:
example = dataset[0]

equation = digitize_equation(example["equation"])

steps = digitize_steps(example["steps"])

print("Equation:", equation)

print("\nDigitized Steps:")

for s in steps:
    print(s)


Equation: x**2 - 6*x

Digitized Steps:
x**2 - 6*x
x*(x - 6)
x - 6
x


In [41]:
digitized_dataset = []

for entry in dataset:

    eq = digitize_equation(entry["equation"])

    steps = digitize_steps(entry["steps"])

    digitized_dataset.append({
        "equation": eq,
        "steps": steps
    })

print("Digitization complete")


Digitization complete


In [42]:
digitized_dataset[0]


{'equation': x**2 - 6*x, 'steps': [x**2 - 6*x, x*(x - 6), x - 6, x]}

In [43]:
expr = digitized_dataset[0]["steps"][1]

sp.expand(expr)


x**2 - 6*x